### After clearing gates:
1. Data Governance: Must data stay private & never leave machine?
    - self hosted/No API
2. Cost Cieling: Zero budget -> every paid API is out.
3. hardware fit --> Does it actually run on my device?
    - Too big to load = disqualified, not "low quality"

The selected model needs to be scored based on:
1. faithfulness
2. Grounding
3. Refusal behaviour
4. Citation / tracibility
5. Controllability and latency


### Selected models

1. Llama 3.2 3B Instruct	3B	128K	Meta Llama Community License
2. Qwen2.5-3B-Instruct	3B	32K	Qwen custom license
3. Microsoft Phi-4-mini-instruct	3.8B	128K	MIT	Strongest small option for reasoning and document QA
4. Ministral 3B Instruct	3B	128K	Mistral Research/custom license
5. IBM Granite 3.3 2B Instruct	2B	128K	Apache 2.0	Good long-context and enterprise-document option

> The above model still needs to be calculate dfor the how much memory they will occupy during runtime and overhead because of kv caching


## # Creating a function to calculate the KV cache size overhead size and the estimate of the model size that fir in the model or not

In [17]:
## I need some component details of the architecture to calculate th =e kv cache size
# Number of kv heads
# Number of layers
# The output dimenion of the model
import requests
from huggingface_hub import ModelCard
def pull_config(model_id):
    url = f"https://huggingface.co/{model_id}/resolve/main/config.json"
    content = requests.get(url).json()
    mechanics = {"model": model_id.split("/")[1],
                 "context_window": content.get('max_position_embeddings',""),
                 "precision": content.get('torch_dtype',""),
                 "specs": {"num_layers": content.get("num_hidden_layers",""),
                           "num_kv_heads": content.get("num_key_value_heads",""),
                           "num_attention_heads": content.get("num_attention_heads",""),
                           "head_size": content.get("hidden_size","")}
                           }
    return mechanics

pull_config("ibm-granite/granite-3.3-2b-instruct")

    

{'model': 'granite-3.3-2b-instruct',
 'context_window': 131072,
 'precision': 'bfloat16',
 'specs': {'num_layers': 40,
  'num_kv_heads': 8,
  'num_attention_heads': 32,
  'head_size': 2048}}

In [ ]:
def calculate_kv_cache_size(model_id, token_sequence_length=8000):
    model_specs = pull_config(model_id)
    if model_specs['precision'] == 'bfloat16':
        bytes_per_number = 2
    elif model_specs['precision'] == 'F32' or model_specs['precision'] == 'FP32':
        bytes_per_number = 4
    elif model_specs['precision'] == 'F8' or model_specs['precision'] == 'FP8':
        bytes_per_number = 1
    else:
        raise ValueError(f"Unknown precision: {model_specs['precision']}")

    num_kv_heads = model_specs['specs']['num_kv_heads']
    num_layers = model_specs['specs']['num_layers']
    head_dim = model_specs['specs']['head_size']/model_specs['specs']['num_attention_heads']

    peak_kv = 2*num_kv_heads*num_layers*head_dim*bytes_per_number*token_sequence_length

    kv_in_MB = peak_kv / (1024*1024)

    return f"{kv_in_MB} MB"

In [41]:
calculate_kv_cache_size("ibm-granite/granite-3.3-2b-instruct")

'625.0 MB'

In [31]:
2048/32

64.0